<img src='images/gdd-logo.png' width='300px' align='right' style="padding: 15px">

# Context Managers

In Python, a context manager is an object that allows you to control the context in which to run code. You can define how the context is created, and then close or "tear down" the context when you are finished.

<a id='resources'></a>
## Managing resources in Python

In any programming language, the usage of resources like files and databases is very common. But it is important to release these resources after usage. Otherwise we can cause memomry issues and risk other unintended side effects.

For example, let's open the following text file and read it in as a string to interact with.

In [1]:
my_file = open("data/example.txt")

In [3]:
text = my_file.read()

In [4]:
print(text)

This is some example text. Yay!


In [5]:
len(text)

31

Notice at the moment, the file is still *open*.

In [6]:
my_file.closed

False

It can cause issues if too many files are open as they take up space in memory.

Uncomment and run the cell below to demonstrate this.

In [7]:
file_descriptors = []
for x in range(100000):
    file_descriptors.append(open("data/example.txt"))

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/Users/pnl1dv1l/GithubRepos/prod-ready-ml-public/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3823, in run_code
  File "/var/folders/fl/6722pk112vscxjf9zgtnf1g40000gp/T/ipykernel_94655/3524280246.py", line 3, in <module>
    file_descriptors.append(open('data/example.txt', 'r'))
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pnl1dv1l/GithubRepos/prod-ready-ml-public/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 339, in _modified_open
OSError: [Errno 24] Too many open files: 'data/example.txt'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/pnl1dv1l/GithubRepos/prod-ready-ml-public/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 2279, in showtraceback
  File "/Users/pnl1dv1l/GithubRepos/prod-ready-ml-public/.venv/lib/python3.12/site-packages/IPython/core/ultratb.p

We get an error message saying that too many files are open. 

*Restart the kernel and continue.*

To avoid situations like above, when we have finished with a file we should close it.

In [7]:
my_file = open("data/example.txt")

In [8]:
my_file.close()

In [9]:
my_file.closed

True

However, it would be very helpful if user have a mechanism for the automatic setup and teardown of resources.

In fact, as the `open()` function is a **context manager**, it can facilitate the proper handling of resources.

The most common way to do so is by using the `with` keyword. As shown below, it allows us to interact with our file by creating creates a *runtime context*, which is then closed afterwards.

In [10]:
with open("data/example.txt") as my_file:
    text = my_file.read()
    length = len(text)
    print(text)
    print(length)

This is some example text. Yay!
31


In [11]:
my_file.closed

True

In [12]:
print(f"The file is {length} characters long and the first word is {text.split()[0]}")

The file is 31 characters long and the first word is This


<a id='caterers'></a>
## Caterers are Context Managers


<img src='images/party.jpeg' width=500px>

Imagine you are hosting a fancy party. You may get caterers to help with the food and refreshments. 

In this situation, the caterers are analogous to the work that context managers do.

|Context Manager|Caterers|
|:---|:---|
|Set up a context|Set up the tables/prepare the food/drinks}
|Run your code|Leave you to party|
|Tear down the context|Clean up the mess|

<a id='use'></a>

## Using a context manager:

To use a context manager you open the context with the keyword `with`. Any code written in the indented block will run in the context.

```python
with <context-manager>(<args>) as <variable-name>:
    # your code here
    # this code is running 'inside the context'
        
# This code runs after the context is removed
```

---

<a id='ex-use'></a>

## <mark>Exercise: Practice using context managers</mark>

For exercise 1 you will need the following information about the data to use:

|File Name|Full Book Name|
|---|---|
|`data/alice.txt`|Alice's Adventures in Wonderland|
|`data/frankenstein.txt`|Frankenstein; or, The Modern Prometheus|
|`data/pride.txt`|Pride and Prejudice|


#### **Exercise 1:** Count how many times Lewis Caroll uses the word rabbit in the first chapter of Alice's Adventures in Wonderland

- Open `"data/alice.txt"` and assign the file to `file`.
- Using `file.read()` assign a new variable text with the contents of `alice.txt`.
- Use the `str.count()` method to count the number of times the word `rabbit` appears.

In [17]:
with open("data/alice.txt") as file:
    content = file.read()
print(content.lower().count("rabbit"))
print(content)

9
Alice’s Adventures in Wonderland


CHAPTER I. Down the Rabbit-Hole

Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into
the book her sister was reading, but it had no pictures or
conversations in it, “and what is the use of a book,” thought Alice
“without pictures or conversations?”

So she was considering in her own mind (as well as she could, for the
hot day made her feel very sleepy and stupid), whether the pleasure of
making a daisy-chain would be worth the trouble of getting up and
picking the daisies, when suddenly a White Rabbit with pink eyes ran
close by her.

There was nothing so _very_ remarkable in that; nor did Alice think it
so _very_ much out of the way to hear the Rabbit say to itself, “Oh
dear! Oh dear! I shall be late!” (when she thought it over afterwards,
it occurred to her that she ought to have wondered at this, but at the
time it all seemed quite natural); but when the Rabbit

In [ ]:
# %load answers/ex-use1.py
with open("data/alice.txt") as file:
    text = file.read()


num_rabbits = text.lower().split().count("rabbit")

print(f'Lewis Carroll uses the word "rabbit" {num_rabbits} times')

## Building your own context managers

There are two ways to build a context manager. With either a `OOP-based` or **`generator-based`** approach. 

### OOP-based context managers

To define custom context managers we need to create a class that implements `__enter__()` and `__exit__()`.

- You can also define `__init__()` to specify arguments that the context manager can take.
- `__exit__()` needs to accept a reference to `self`, the type of exception it might throw, the exception itself and a traceback object as arguments.

In [18]:
import sqlite3

import pandas as pd


class DBConnection:
    def __init__(self, db_name):
        self.db = db_name

    def __enter__(self):
        self.conn = sqlite3.connect(self.db)
        return self.conn

    def __exit__(self, exc_class, exc, traceback):
        self.conn.close()


with DBConnection("SQLDatabase.db") as db:
    trends = pd.read_sql("""SELECT * FROM programming_trends""", db)

trends.head()

,Month,Python,SQL,R,JavaScript,Visual_Basic_for_Applications
0,2004-01-01,14,84,6,88,14
1,2004-02-01,14,94,6,95,16
2,2004-03-01,15,93,<5,89,15
3,2004-04-01,14,96,7,90,15
4,2004-05-01,12,91,6,87,14


#### **Exercise 2:** Create a context manager called `InDir` that allows you to run code form a different directory than the current working directory.

In [19]:
!pwd

/Users/pnl1dv1l/GithubRepos/prod-ready-ml-public/notebooks/context_managers


In [22]:
import os


class InDir:
    def __init__(self, directory):
        self.directory = directory

    def __enter__(self):
        self.original_directory = os.getcwd()
        os.chdir(self.directory)
        return self.directory

    def __exit__(self, exc_class, exc, traceback):
        os.chdir(self.original_directory)

In [23]:
# This code should work unedited
with InDir("../../"):
    notebook_files = os.listdir("notebooks")

notebook_files

['decorators_pipelines',
 'oop',
 '02_code_quality.ipynb',
 'cli',
 'context_managers',
 '06_logging.ipynb',
 'fastapi',
 '04_unit_tests.ipynb',
 '05_more_testing.ipynb',
 '01_packaging.ipynb',
 '07_hackathon.ipynb',
 'generators_iterators',
 '03_quality_checks.ipynb']

In [24]:
!pwd # Should not have changed!

/Users/pnl1dv1l/GithubRepos/prod-ready-ml-public/notebooks/context_managers


In [ ]:
# %load answers/ex-build1.py
import os


class InDir:
    def __init__(self, path):
        self.old_path = os.getcwd()
        self.path = path

    def __enter__(self):
        os.chdir(self.path)

    def __exit__(self, exc_class, exc, traceback):
        os.chdir(self.old_path)


with InDir("../../"):
    notebook_files = os.listdir("notebooks")

notebook_files

#### **Exercise 3:** Create a context manager called `Timer` that times the execution time of code in its body.

- You can also add an extra argument that allows the user to add a description to the log of the execution time.

In [26]:
import time


class Timer:
    def __init__(self, name):
        self.name = name
        self.begin = time.time()

    def __enter__(self):
        return self

    def __exit__(self, exc_class, exc, traceback):
        self.end = time.time()
        self.interval = self.end - self.begin
        print(f"{self.name}: {self.interval} seconds")

In [ ]:
# %load answers/ex-build2.py


class Timer:
    def __init__(self, description):
        self.description = description

    def __enter__(self):
        self.start = time.time()

    def __exit__(self, type, value, traceback):
        self.end = time.time()
        print(f"{self.description}: {self.end - self.start}")


with Timer("Timed-time:"):
    time.sleep(2)

### Generator-based context managers

Instead of creating context managers by designing classes, it's usually more ergonomic and idiomatic to use a generator function.

You can decorate any generator with the `@contextlib.contextmanager` decorator. The code before the `yield` statemt will act as the context set-up. The `yield` statement can return a handle to the object created in the context, and any code after `yield` will act as teardowm.

The quivalent to the previous database context manager would be:

In [28]:
import contextlib

import pandas as pd


@contextlib.contextmanager
def my_database(db_name):

    conn = sqlite3.connect(db_name)

    yield conn

    conn.close()


with my_database("SQLDatabase.db") as db:
    trends = pd.read_sql("""SELECT * FROM programming_trends""", db)

trends.head()

,Month,Python,SQL,R,JavaScript,Visual_Basic_for_Applications
0,2004-01-01,14,84,6,88,14
1,2004-02-01,14,94,6,95,16
2,2004-03-01,15,93,<5,89,15
3,2004-04-01,14,96,7,90,15
4,2004-05-01,12,91,6,87,14


Python's `sqlite` package actually comes with it's own context manager, great! So we can use that instead. The above demonstrates the flow of using a context manager while connecting to SQL.

In [29]:
with sqlite3.connect("SQLDatabase.db") as conn:
    query = """SELECT * FROM programming_trends"""
    results = conn.execute(query).fetchall()

results

[('2004-01-01', 14, 84, 6, 88, 14),
 ('2004-02-01', 14, 94, 6, 95, 16),
 ('2004-03-01', 15, 93, '<5', 89, 15),
 ('2004-04-01', 14, 96, 7, 90, 15),
 ('2004-05-01', 12, 91, 6, 87, 14),
 ('2004-06-01', 14, 100, 6, 89, 17),
 ('2004-07-01', 15, 98, 6, 86, 16),
 ('2004-08-01', 15, 94, '<5', 88, 17),
 ('2004-09-01', 15, 88, 6, 80, 16),
 ('2004-10-01', 15, 89, 7, 77, 14),
 ('2004-11-01', 13, 84, 6, 73, 14),
 ('2004-12-01', 14, 84, 6, 72, 14),
 ('2005-01-01', 14, 82, 7, 69, 14),
 ('2005-02-01', 14, 86, 7, 74, 15),
 ('2005-03-01', 14, 86, 7, 71, 15),
 ('2005-04-01', 13, 85, 7, 72, 14),
 ('2005-05-01', 14, 83, 7, 70, 14),
 ('2005-06-01', 15, 89, 7, 73, 16),
 ('2005-07-01', 14, 82, 7, 68, 14),
 ('2005-08-01', 14, 80, 6, 70, 15),
 ('2005-09-01', 13, 79, 7, 67, 13),
 ('2005-10-01', 15, 76, 7, 66, 13),
 ('2005-11-01', 14, 78, 7, 63, 13),
 ('2005-12-01', 13, 71, 7, 62, 13),
 ('2006-01-01', 13, 71, 7, 59, 12),
 ('2006-02-01', 14, 78, 8, 64, 13),
 ('2006-03-01', 13, 78, 7, 65, 13),
 ('2006-04-01', 13, 7

**Caveat**: Often you would want to include a `try` (`except`) and `finally` within the function to ensure you are able to handle any connection errors you might have.

#### **Exercise 4:** Recreate the context managers from exercises 2 and 3 with generators

In [32]:
import time


@contextlib.contextmanager
def timer(description):

    conn = time.time()

    yield conn

    print(f"{description}: {time.time() - conn} seconds")

In [33]:
# %load answers/ex-convert2.py
import contextlib
import os
import time


@contextlib.contextmanager
def in_dir(path):
    old_path = os.getcwd()
    os.chdir(path)
    yield
    os.chdir(old_path)


with in_dir("../../"):
    notebook_files = os.listdir("notebooks")

notebook_files


@contextlib.contextmanager
def timer(description):
    start = time.time()
    yield
    end = time.time()
    print(f"{description}: {end - start}")


with timer("timed-time"):
    time.sleep(1.5)